# PyHealth Transformer Baseline - Google Colab

This notebook is intended to be ran within Google Colab (using A100 runtime) to test validity of the synthetic EHR generation implementation within PyHealth.
It runs the equivalent of the **transformer_baseline** mode from [Chufan's baselines.py](https://github.com/chufangao/reproducible_synthetic_ehr/blob/main/baselines.py), but using the implemented PyHealth infrastructure.
The results of the two workflows are then compared. It will take ~1-1.5 hours to run this full notebook within Colab.


**What this does:**
1. Processes MIMIC data into sequential format
2. Trains a GPT-2 style transformer on diagnosis sequences
3. Generates synthetic patient histories
4. Compares with original transformer_baseline outputs

**Prerequisites:**
- Original transformer_baseline results already in Google Drive
- MIMIC-III data files
- Train/test patient ID files

## Step 1: Setup & Check GPU

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive/

## Step 2: Install Dependencies

In [ ]:
import os

# Where to clone from
clone_repo = "https://github.com/ethanrasmussen/PyHealth.git"
clone_branch = "implement_baseline"

# Where to save repo/package
repo_dir = "/content/PyHealth"

if not os.path.exists(repo_dir):
    !git clone -b {clone_branch} {clone_repo} {repo_dir}
%cd /content/PyHealth

# install your repo without letting pip touch torch/cuda stack
%pip install -e . --no-deps

# now install the runtime deps you actually need for this notebook
%pip install -U --no-cache-dir --force-reinstall "numpy==2.2.0"
%pip install -U "transformers==4.53.2" "tokenizers" "accelerate" "peft"
%pip install -U "pandas" "tqdm"

## Step 3: Configure Paths

**IMPORTANT:** Update these paths to match your Google Drive structure!

In [ ]:
# ========================================
# CONFIGURE YOUR PATHS HERE
# ========================================

# Input data paths
MIMIC_DATA_PATH = "/content/drive/MyDrive/mimic3_data/"
TRAIN_PATIENTS_PATH = "/content/drive/MyDrive/mimic3_data/train_patient_ids.txt"
TEST_PATIENTS_PATH = "/content/drive/MyDrive/mimic3_data/test_patient_ids.txt"

# Original transformer_baseline output (for comparison)
ORIGINAL_OUTPUT_CSV = "/content/drive/MyDrive/original_output/transformer_baseline/transformer_baseline_synthetic_ehr.csv"

# PyHealth output directory
PYHEALTH_OUTPUT = "/content/pyhealth_transformer_output"

# Training hyperparameters
NUM_EPOCHS = 50
TRAIN_BATCH_SIZE = 64
GEN_BATCH_SIZE = 512
NUM_SYNTHETIC_SAMPLES = 10000
MAX_SEQ_LENGTH = 512

# Model architecture
EMBEDDING_DIM = 512
NUM_LAYERS = 8
NUM_HEADS = 8

print("Configuration:")
print(f"  MIMIC Data: {MIMIC_DATA_PATH}")
print(f"  Train IDs: {TRAIN_PATIENTS_PATH}")
print(f"  Test IDs: {TEST_PATIENTS_PATH}")
print(f"  Original output: {ORIGINAL_OUTPUT_CSV}")
print(f"  PyHealth output: {PYHEALTH_OUTPUT}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {device}")

In [ ]:
# Verify files exist
required_files = [
    os.path.join(MIMIC_DATA_PATH, "ADMISSIONS.csv"),
    os.path.join(MIMIC_DATA_PATH, "PATIENTS.csv"),
    os.path.join(MIMIC_DATA_PATH, "DIAGNOSES_ICD.csv"),
    TRAIN_PATIENTS_PATH,
    TEST_PATIENTS_PATH,
]

print("Checking required files:")
all_exist = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✓" if exists else "✗"
    print(f"  {status} {f}")
    if not exists:
        all_exist = False

# Check original output
original_exists = os.path.exists(ORIGINAL_OUTPUT_CSV)
print(f"\nOriginal transformer_baseline output:")
print(f"  {'✓' if original_exists else '✗'} {ORIGINAL_OUTPUT_CSV}")

if all_exist:
    print("\n✓ All MIMIC files found!")
    if original_exists:
        print("✓ Original output found - will compare after generation")
    else:
        print("⚠️  Original output not found - will skip comparison")
else:
    print("\n✗ Some files are missing. Please update paths.")

## Step 4: Process MIMIC Data

This processes MIMIC data the same way as the original baselines.py

In [ ]:
import pandas as pd
from pyhealth.synthetic_ehr_utils.synthetic_ehr_utils import process_mimic_for_generation

print("Processing MIMIC data...")

# Process data
data = process_mimic_for_generation(
    mimic_data_path=MIMIC_DATA_PATH,
    train_patients_path=TRAIN_PATIENTS_PATH,
    test_patients_path=TEST_PATIENTS_PATH,
)

train_ehr = data["train_ehr"]
test_ehr = data["test_ehr"]
train_sequences = data["train_sequences"]
test_sequences = data["test_sequences"]

print("\n" + "="*80)
print("Data Processing Complete")
print("="*80)
print(f"Train EHR shape: {train_ehr.shape}")
print(f"Test EHR shape: {test_ehr.shape}")
print(f"Train sequences: {len(train_sequences)}")
print(f"Test sequences: {len(test_sequences)}")

# Check max sequence length
max_len_train = max([len(seq.split()) for seq in train_sequences])
print(f"\nMax sequence length in training data: {max_len_train}")

print("\nSample sequence (first patient):")
print(train_sequences[0][:200] + "...")

## Step 5: Build Custom Tokenizer

Build a word-level tokenizer on the medical codes (same as original)

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, processors
from transformers import PreTrainedTokenizerFast

print("Building custom tokenizer...")

# Use WordLevel model (treats each code as a single token)
tokenizer_obj = Tokenizer(models.WordLevel(unk_token="[UNK]"))
tokenizer_obj.pre_tokenizer = pre_tokenizers.Whitespace()

# Special tokens
special_tokens = ["[UNK]", "[PAD]", "[BOS]", "[EOS]"]
trainer = trainers.WordLevelTrainer(special_tokens=special_tokens)

# Train tokenizer on sequences
tokenizer_obj.train_from_iterator(train_sequences, trainer=trainer)

# Add post-processing to add BOS/EOS automatically
tokenizer_obj.post_processor = processors.TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", tokenizer_obj.token_to_id("[BOS]")),
        ("[EOS]", tokenizer_obj.token_to_id("[EOS]")),
    ],
)

# Wrap in HuggingFace tokenizer
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_obj,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
)

vocab_size = len(tokenizer)
print(f"\n✓ Tokenizer built")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Special tokens: {special_tokens}")
print(f"  BOS token ID: {tokenizer.bos_token_id}")
print(f"  EOS token ID: {tokenizer.eos_token_id}")
print(f"  PAD token ID: {tokenizer.pad_token_id}")

# Test tokenization
test_seq = train_sequences[0]
encoded = tokenizer(test_seq, truncation=True, max_length=MAX_SEQ_LENGTH)
print(f"\nTest encoding (first 20 tokens): {encoded['input_ids'][:20]}")
decoded = tokenizer.decode(encoded['input_ids'][:20], skip_special_tokens=False)
print(f"Decoded: {decoded}")

## Step 6: Create PyTorch Dataset

In [ ]:
from torch.utils.data import Dataset

class EHRDataset(Dataset):
    def __init__(self, txt_list, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.input_ids = []
        
        print(f"Tokenizing {len(txt_list)} sequences...")
        for txt in txt_list:
            encodings = tokenizer(
                txt,
                truncation=True,
                max_length=max_length,
                padding="max_length"
            )
            self.input_ids.append(torch.tensor(encodings["input_ids"]))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "labels": self.input_ids[idx]}

# Create dataset
train_dataset = EHRDataset(train_sequences, tokenizer, max_length=MAX_SEQ_LENGTH)

print(f"\n✓ Dataset created")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")

## Step 7: Initialize GPT-2 Model

Create a GPT-2 style decoder model (same architecture as original)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

print("Initializing GPT-2 model...")

# Configure model
config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=MAX_SEQ_LENGTH,
    n_ctx=MAX_SEQ_LENGTH,
    n_embd=EMBEDDING_DIM,
    n_layer=NUM_LAYERS,
    n_head=NUM_HEADS,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = GPT2LMHeadModel(config).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())

print(f"\n✓ Model initialized")
print(f"  Total parameters: {num_params:,}")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Num layers: {NUM_LAYERS}")
print(f"  Num heads: {NUM_HEADS}")
print(f"  Device: {device}")

## Step 8: Train Model

Train using HuggingFace Trainer (same as original)

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

print("Setting up training...")

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal Language Modeling (not masked)
)

# Training arguments
training_args = TrainingArguments(
    output_dir=os.path.join(PYHEALTH_OUTPUT, "checkpoints"),
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    logging_steps=100,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    save_total_limit=2,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print(f"\nStarting training for {NUM_EPOCHS} epochs...")
print(f"This will take approximately {NUM_EPOCHS * 2} minutes with GPU")
print(f"Batch size: {TRAIN_BATCH_SIZE}")
print(f"Total steps: {len(train_dataset) // TRAIN_BATCH_SIZE * NUM_EPOCHS}")
print("\n" + "="*80)

# Train!
trainer.train()

print("\n" + "="*80)
print("✓ Training complete!")
print("="*80)

In [ ]:
# Save model
os.makedirs(PYHEALTH_OUTPUT, exist_ok=True)
model_save_path = os.path.join(PYHEALTH_OUTPUT, "transformer_baseline_model_final")
trainer.save_model(model_save_path)

print(f"✓ Model saved to: {model_save_path}")

## Step 9: Generate Synthetic EHRs

Generate synthetic patient histories using the trained model

In [ ]:
from tqdm import trange
from pyhealth.synthetic_ehr_utils.synthetic_ehr_utils import sequences_to_tabular

print("\n" + "="*80)
print("Generating Synthetic EHRs")
print("="*80)
print(f"Target samples: {NUM_SYNTHETIC_SAMPLES}")
print(f"Batch size: {GEN_BATCH_SIZE}")
print(f"Max length: {max_len_train}\n")

model.eval()

all_syn_dfs = []
start_patient_id = 0

for start_idx in trange(0, NUM_SYNTHETIC_SAMPLES, GEN_BATCH_SIZE, desc="Generating"):
    end_idx = min(start_idx + GEN_BATCH_SIZE, NUM_SYNTHETIC_SAMPLES)
    batch_size = end_idx - start_idx
    
    # Prepare batch of BOS tokens
    batch_input_ids = torch.tensor([[tokenizer.bos_token_id]] * batch_size).to(device)
    
    # Generate sequences
    with torch.no_grad():
        generated_ids = model.generate(
            batch_input_ids,
            max_length=max_len_train,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode to text
    all_decoded = []
    for sample in generated_ids:
        decoded = tokenizer.decode(sample, skip_special_tokens=True)
        all_decoded.append(decoded)
    
    # Convert to tabular
    syn_df = sequences_to_tabular(all_decoded)
    syn_df['SUBJECT_ID'] += start_patient_id
    start_patient_id += batch_size
    all_syn_dfs.append(syn_df)

# Combine all batches
all_syn_df = pd.concat(all_syn_dfs, ignore_index=True)

print("\n" + "="*80)
print("Generation Complete!")
print("="*80)
print(f"Generated patients: {all_syn_df['SUBJECT_ID'].nunique()}")
print(f"Total visits: {all_syn_df['HADM_ID'].nunique()}")
print(f"Total codes: {len(all_syn_df)}")
print(f"Unique codes: {all_syn_df['ICD9_CODE'].nunique()}")
print(f"Avg codes per patient: {len(all_syn_df) / all_syn_df['SUBJECT_ID'].nunique():.2f}")

print("\nFirst 10 rows:")
print(all_syn_df.head(10))

In [ ]:
# Save synthetic data
synthetic_csv_path = os.path.join(PYHEALTH_OUTPUT, "transformer_baseline_synthetic_ehr.csv")
all_syn_df.to_csv(synthetic_csv_path, index=False)

print(f"✓ Synthetic data saved to: {synthetic_csv_path}")

## Step 10: Visualize Synthetic Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Codes per patient
codes_per_patient = all_syn_df.groupby('SUBJECT_ID').size()
axes[0, 0].hist(codes_per_patient, bins=50, edgecolor='black')
axes[0, 0].set_xlabel('Number of codes per patient')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Codes per Patient')

# 2. Visits per patient
visits_per_patient = all_syn_df.groupby('SUBJECT_ID')['HADM_ID'].nunique()
axes[0, 1].hist(visits_per_patient, bins=30, edgecolor='black')
axes[0, 1].set_xlabel('Number of visits per patient')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Visits per Patient')

# 3. Top codes
top_codes = all_syn_df['ICD9_CODE'].value_counts().head(20)
axes[1, 0].barh(range(len(top_codes)), top_codes.values)
axes[1, 0].set_yticks(range(len(top_codes)))
axes[1, 0].set_yticklabels(top_codes.index, fontsize=8)
axes[1, 0].set_xlabel('Frequency')
axes[1, 0].set_title('Top 20 Most Frequent Codes')
axes[1, 0].invert_yaxis()

# 4. Codes per visit
codes_per_visit = all_syn_df.groupby(['SUBJECT_ID', 'HADM_ID']).size()
axes[1, 1].hist(codes_per_visit, bins=30, edgecolor='black')
axes[1, 1].set_xlabel('Number of codes per visit')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Codes per Visit')

plt.tight_layout()
plt.savefig(os.path.join(PYHEALTH_OUTPUT, 'synthetic_visualization.png'), dpi=150)
plt.show()

print(f"✓ Visualization saved")

## Step 11: Compare with Original Transformer Baseline

Compare PyHealth results with your original transformer_baseline outputs

In [ ]:
# Check if original file exists
if os.path.exists(ORIGINAL_OUTPUT_CSV):
    print("✓ Original output found - running comparison...\n")
    COMPARE = True
    
    # Load original data
    original_df = pd.read_csv(ORIGINAL_OUTPUT_CSV)
    pyhealth_df = all_syn_df
    
    print("Loaded datasets:")
    print(f"  Original shape: {original_df.shape}")
    print(f"  PyHealth shape: {pyhealth_df.shape}")
else:
    print("✗ Original output not found - skipping comparison")
    print(f"Expected at: {ORIGINAL_OUTPUT_CSV}")
    COMPARE = False

In [ ]:
if COMPARE:
    print("\n" + "="*80)
    print("STATISTICAL COMPARISON")
    print("="*80)
    
    # Basic statistics
    comparison_stats = pd.DataFrame({
        'Metric': [
            'Total patients',
            'Total visits',
            'Total codes',
            'Unique codes',
            'Avg codes/patient',
            'Avg visits/patient',
            'Avg codes/visit'
        ],
        'Original': [
            original_df['SUBJECT_ID'].nunique(),
            original_df.groupby('SUBJECT_ID')['HADM_ID'].nunique().sum(),
            len(original_df),
            original_df['ICD9_CODE'].nunique(),
            f"{len(original_df) / original_df['SUBJECT_ID'].nunique():.2f}",
            f"{original_df.groupby('SUBJECT_ID')['HADM_ID'].nunique().mean():.2f}",
            f"{original_df.groupby(['SUBJECT_ID', 'HADM_ID']).size().mean():.2f}"
        ],
        'PyHealth': [
            pyhealth_df['SUBJECT_ID'].nunique(),
            pyhealth_df.groupby('SUBJECT_ID')['HADM_ID'].nunique().sum(),
            len(pyhealth_df),
            pyhealth_df['ICD9_CODE'].nunique(),
            f"{len(pyhealth_df) / pyhealth_df['SUBJECT_ID'].nunique():.2f}",
            f"{pyhealth_df.groupby('SUBJECT_ID')['HADM_ID'].nunique().mean():.2f}",
            f"{pyhealth_df.groupby(['SUBJECT_ID', 'HADM_ID']).size().mean():.2f}"
        ]
    })
    
    print(comparison_stats.to_string(index=False))

In [ ]:
if COMPARE:
    from scipy import stats
    
    print("\n" + "="*80)
    print("DISTRIBUTION COMPARISON")
    print("="*80)
    
    # Code frequency correlation
    orig_freq = original_df['ICD9_CODE'].value_counts()
    pyh_freq = pyhealth_df['ICD9_CODE'].value_counts()
    
    # Get common codes
    common_codes = set(orig_freq.index) & set(pyh_freq.index)
    print(f"\nCommon codes: {len(common_codes)}")
    print(f"Original-only codes: {len(set(orig_freq.index) - common_codes)}")
    print(f"PyHealth-only codes: {len(set(pyh_freq.index) - common_codes)}")
    
    if len(common_codes) > 0:
        orig_common = orig_freq[list(common_codes)]
        pyh_common = pyh_freq[list(common_codes)]
        
        # Calculate correlation
        correlation = orig_common.corr(pyh_common)
        print(f"\nCode frequency correlation (Pearson): {correlation:.4f}")
        
        # KS test on distributions
        codes_per_patient_orig = original_df.groupby('SUBJECT_ID').size()
        codes_per_patient_pyh = pyhealth_df.groupby('SUBJECT_ID').size()
        ks_stat, ks_pval = stats.ks_2samp(codes_per_patient_orig, codes_per_patient_pyh)
        print(f"\nKS test (codes per patient):")
        print(f"  Statistic: {ks_stat:.4f}")
        print(f"  P-value: {ks_pval:.4f}")
        print(f"  Significant difference: {'Yes' if ks_pval < 0.05 else 'No'}")

In [ ]:
if COMPARE:
    print("\n" + "="*80)
    print("VISUAL COMPARISON")
    print("="*80)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Codes per patient comparison
    codes_per_patient_orig = original_df.groupby('SUBJECT_ID').size()
    codes_per_patient_pyh = pyhealth_df.groupby('SUBJECT_ID').size()
    
    axes[0, 0].hist(codes_per_patient_orig, bins=50, alpha=0.7, label='Original', edgecolor='black')
    axes[0, 0].hist(codes_per_patient_pyh, bins=50, alpha=0.7, label='PyHealth', edgecolor='black')
    axes[0, 0].set_xlabel('Codes per patient')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution: Codes per Patient')
    axes[0, 0].legend()
    
    # 2. Visits per patient comparison
    visits_per_patient_orig = original_df.groupby('SUBJECT_ID')['HADM_ID'].nunique()
    visits_per_patient_pyh = pyhealth_df.groupby('SUBJECT_ID')['HADM_ID'].nunique()
    
    axes[0, 1].hist(visits_per_patient_orig, bins=30, alpha=0.7, label='Original', edgecolor='black')
    axes[0, 1].hist(visits_per_patient_pyh, bins=30, alpha=0.7, label='PyHealth', edgecolor='black')
    axes[0, 1].set_xlabel('Visits per patient')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution: Visits per Patient')
    axes[0, 1].legend()
    
    # 3. Code frequency correlation scatter
    if len(common_codes) > 0:
        axes[1, 0].scatter(orig_common, pyh_common, alpha=0.5)
        max_val = max(orig_common.max(), pyh_common.max())
        axes[1, 0].plot([0, max_val], [0, max_val], 'r--', label='Perfect match')
        axes[1, 0].set_xlabel('Original frequency')
        axes[1, 0].set_ylabel('PyHealth frequency')
        axes[1, 0].set_title(f'Code Frequency Correlation (r={correlation:.3f})')
        axes[1, 0].legend()
        axes[1, 0].set_xscale('log')
        axes[1, 0].set_yscale('log')
    
    # 4. Top codes comparison
    top_n = 15
    top_orig = orig_freq.head(top_n)
    top_pyh = pyh_freq.head(top_n)
    
    x = range(top_n)
    width = 0.35
    axes[1, 1].bar([i - width/2 for i in x], top_orig.values, width, label='Original', alpha=0.8)
    axes[1, 1].bar([i + width/2 for i in x], top_pyh.values, width, label='PyHealth', alpha=0.8)
    axes[1, 1].set_xlabel('Top codes (rank)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title(f'Top {top_n} Most Frequent Codes')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(PYHEALTH_OUTPUT, 'comparison_visualization.png'), dpi=150)
    plt.show()
    
    print(f"\n✓ Comparison visualization saved")

In [ ]:
if COMPARE:
    print("\n" + "="*80)
    print("VALIDATION CHECKS")
    print("="*80)
    
    checks = []
    
    # Check 1: Similar number of patients
    orig_patients = original_df['SUBJECT_ID'].nunique()
    pyh_patients = pyhealth_df['SUBJECT_ID'].nunique()
    patients_diff = abs(orig_patients - pyh_patients) / orig_patients
    checks.append(('Similar number of patients (within 5%)', patients_diff < 0.05))
    
    # Check 2: Similar total codes
    orig_total = len(original_df)
    pyh_total = len(pyhealth_df)
    total_diff = abs(orig_total - pyh_total) / orig_total
    checks.append(('Similar total codes (within 20%)', total_diff < 0.20))
    
    # Check 3: Similar codes per patient
    orig_cpp = len(original_df) / original_df['SUBJECT_ID'].nunique()
    pyh_cpp = len(pyhealth_df) / pyhealth_df['SUBJECT_ID'].nunique()
    cpp_diff = abs(orig_cpp - pyh_cpp) / orig_cpp
    checks.append(('Similar codes per patient (within 20%)', cpp_diff < 0.20))
    
    # Check 4: High frequency correlation
    if 'correlation' in locals():
        checks.append(('High code frequency correlation (>0.7)', correlation > 0.7))
    
    # Print results
    print()
    for check_name, passed in checks:
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"  {status} - {check_name}")
    
    passed_count = sum([c[1] for c in checks])
    total_count = len(checks)
    
    print(f"\nResult: {passed_count}/{total_count} checks passed")
    
    if passed_count == total_count:
        print("\n🎉 All checks passed! PyHealth implementation matches original.")
    elif passed_count >= total_count * 0.75:
        print("\n✓ Most checks passed. Minor differences are expected due to randomness.")
    else:
        print("\n⚠️  Some checks failed. Review the distributions above.")

## Step 12: Download Results

In [ ]:
from google.colab import files
import shutil

# Create zip with all outputs
output_zip = '/content/pyhealth_transformer_results.zip'
shutil.make_archive(
    output_zip.replace('.zip', ''),
    'zip',
    PYHEALTH_OUTPUT
)

print(f"Created: {output_zip}")
print("Downloading...")

files.download(output_zip)

print("✓ Download complete!")

## Summary

### What You Accomplished:

1. ✓ Processed MIMIC data into sequences
2. ✓ Built custom word-level tokenizer
3. ✓ Trained GPT-2 style transformer model
4. ✓ Generated synthetic patient histories
5. ✓ Compared with original transformer_baseline

### Files Generated:

- `transformer_baseline_synthetic_ehr.csv` - Synthetic data
- `transformer_baseline_model_final/` - Trained model
- `synthetic_visualization.png` - Data plots
- `comparison_visualization.png` - Comparison plots

### Key Metrics:

Check the comparison section above to see if:
- Similar number of patients generated
- Similar code distributions
- High correlation in code frequencies
- Similar visit patterns

If all checks passed, the PyHealth implementation is working correctly! 🎉